# Exercise 9.1 — Variable importance, and a column of pure noise (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day9/day6-01-importance.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

Day 5 ended with a random forest that predicts diabetes well but cannot be
read like a tree or a logistic regression. The first question everyone asks of
such a model is: **which variables does it use?** This exercise answers it in
two ways, and then tests both with a column we know to be useless.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

------------------------------------------------------------------------

### Part 1 — The forest

The same 750 people and five predictors as Day 5, from
[`data/diabetes_clean.csv`](data/diabetes_clean.csv): number of pregnancies,
glucose, BMI (`mass`), the diabetes pedigree function and age. As always, 80%
of the people are used for training and 20% are held back as the test set.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

train, test = train_test_split(diabetes, test_size=0.2, random_state=42)

A random forest of 500 trees, as on Day 5. scikit-learn keeps a record of how
much each variable helped while it grows the trees; Part 2 reads it. A forest
is random, so fix `random_state`.

In [ ]:
rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf = rf.fit(train[predictors], train["diabetes"])

test_prob = rf.predict_proba(test[predictors])[:, 1]    # probability of diabetes

round(roc_auc_score(test["diabetes"], test_prob), 3)

The test AUC is 0.794. This is the model we explain all day. R grows
different random trees, so its numbers differ slightly.

------------------------------------------------------------------------

### Part 2 — Impurity importance

**Impurity importance** adds up, for each variable, how much purer its splits
made the training rows, over all the trees. The forest works it out while it
is fitted, so it is free. scikit-learn stores it in
`feature_importances_`, one number per predictor, already scaled to add up
to 1.

In [ ]:
impurity_importance = pd.Series(rf.feature_importances_, index=predictors)
impurity_importance.sort_values(ascending=False).round(3)

Glucose has 0.318 of the total, then BMI 0.205, pedigree 0.198, age 0.169
and pregnancies 0.109. Glucose comes first by a long way.

------------------------------------------------------------------------

### Part 3 — Permutation importance

**Permutation importance** asks a different question: *how much worse is the
model on the test rows if one variable is scrambled?* Shuffle one column so
its values land on the wrong people, predict again, and see how much the AUC
drops. The model is not refitted.

> **Stuck?**
>
> `rng.permutation()` returns the same glucose values in a random order. Then
> predict on the shuffled test rows, `test_shuffled`.

> **Worked answer**

In [ ]:
rng = np.random.default_rng(1)
test_shuffled = test.copy()
test_shuffled["glucose"] = rng.permutation(test_shuffled["glucose"].values)

shuffled_prob = rf.predict_proba(test_shuffled[predictors])[:, 1]
round(roc_auc_score(test_shuffled["diabetes"], shuffled_prob), 3)

> With glucose shuffled the AUC falls from 0.794 to 0.622, a drop of 0.172.
> Run it with another seed and the drop changes a little.

One shuffle is one random draw, so in Part 4 we repeat it 20 times for each
variable and take the average drop.

------------------------------------------------------------------------

### Part 4 — A control: a column of pure noise

Which measure should we believe? Test both on something whose answer we
already know: [`data/diabetes_noise.csv`](data/diabetes_noise.csv) is the same
file with one more column, `noise`: random numbers unrelated to anything. Its
true importance is **zero**.

Read it, keep `noise` as a sixth predictor, and split it exactly as before.
The same `random_state` gives the same people in the test set.

In [ ]:
diabetes_noise = pd.read_csv("data/diabetes_noise.csv")
diabetes_noise["diabetes"] = (diabetes_noise["diabetes"] == "pos").astype(int)

predictors_noise = predictors + ["noise"]

train_noise, test_noise = train_test_split(diabetes_noise, test_size=0.2, random_state=42)

Fit the same forest on these training rows and look at its impurity
importance.

> **Stuck?**
>
> The forest now uses all six predictors, `predictors_noise`.

> **Worked answer**

In [ ]:
rf_noise = RandomForestClassifier(n_estimators=500, random_state=1)
rf_noise = rf_noise.fit(train_noise[predictors_noise], train_noise["diabetes"])

pd.Series(rf_noise.feature_importances_, index=predictors_noise).sort_values(ascending=False).round(3)

> `noise` gets 0.154 of the total, level with the pedigree function (0.154),
> and ahead of age (0.140) and pregnancies (0.104). By this measure, random
> numbers are a respectable predictor.

Now the permutation importance of the same forest on the test rows.
`permutation_importance()` does what you did in Part 3, 20 times for each of
the six variables, with `scoring="roc_auc"`.

In [ ]:
perm = permutation_importance(rf_noise, test_noise[predictors_noise],
                              test_noise["diabetes"], scoring="roc_auc",
                              n_repeats=20, random_state=1)

perm_importance = pd.DataFrame({"variable": predictors_noise,
                                "importance": perm.importances_mean,
                                "sd": perm.importances_std})
perm_importance = perm_importance.sort_values("importance", ascending=False)
perm_importance.round(3)

In [ ]:
ordered = perm_importance.sort_values("importance")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(ordered["variable"], ordered["importance"], xerr=ordered["sd"],
        color="coral", capsize=3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean drop in test AUC when the variable is shuffled (± 1 SD)")
ax.set_title("Random forest: permutation importance")
fig.tight_layout()
plt.show()

`noise` comes out at −0.005, with a standard deviation of 0.008: zero, as it
should. Glucose (0.150), BMI (0.031) and age (0.029) clearly beat it.
Pregnancies (0.012) and the pedigree function (0.006) are within about one
standard deviation of it.

Why the difference? Deep trees always find *some* split on a column with many
different values that tidies up a few training rows. Impurity importance
counts that; permutation importance on new people does not. So a variable that
does not clearly beat `noise` should not be called important.

> **Key points**
>
> - Impurity importance is free and quick, but it is computed on the training rows and is biased: it gave pure noise 15% of the total.
> - Permutation importance is the drop in test AUC when one column is shuffled. Repeat the shuffle and report the mean and spread.
> - A noise column is a control: its true importance is zero, and permutation importance puts it there.